# 🎬 LTX-Video 2.3 Studio on Lightning.ai (GGUF High-Quality)

Welcome! This notebook runs the high-quality **22-Billion parameter LTX-Video GGUF Q8 diffusion model** on Lightning.ai's GPU Studios (e.g., L4 or L40S). 

The Lightning setup uses Q8 diffusion with a smaller stable-diffusion.cpp-compatible GGUF text encoder, then chooses GPU or CPU-offload mode based on available VRAM.

### **Why is this setup different?**
1. **Persistent Files**: Unlike Kaggle, files inside `/teamspace/studios/this_studio/` are saved permanently. You only download the models **once**! 
2. **Zero-Setup Duplication**: If you run this from a published template, all models are pre-cached, and you can skip straight to launching the server and UI.

### 🛠️ Step 1: Install Requirements

In [ ]:
import sys
import os

# Add repository root to system path
sys.path.append(os.path.abspath(".."))

# Install python requirements (minimal Gradio and requests)
print("📥 Checking python requirements...")
!pip install -q -r ../requirements.txt

### 📥 Step 2: Download Model Weights & Binary

Downloads the high-quality GGUF Q8 diffusion model, matching GGUF text encoder, embeddings connector, and VAEs to your persistent `/teamspace/studios/this_studio/models` folder.

In [ ]:
from src.downloader import restore_lightning_binary, download_models

# 1. Download and cache the prebuilt Lightning CUDA engine binary to persistent directory
restore_lightning_binary(target_dir="/teamspace/studios/this_studio/sd_bin")

# 2. Download high-quality LTX-Video GGUF Q8 models persistently
download_models(
    preset="LTX-Video-2.3-FP8", 
    models_base="/teamspace/studios/this_studio/models"
)

### ⚡ Step 3: Start the Inference Server

Spawns the background server using GPU mode for high-VRAM Lightning machines. Use CPU offload only on smaller GPUs such as L4.

In [ ]:
from src.server import start_server, tail_logs

# Start C++ backend server using persistent model directory
server_process = start_server(
    preset="LTX-Video-2.3-FP8",
    bin_path="/teamspace/studios/this_studio/sd_bin/bin/sd-server",
    models_base="/teamspace/studios/this_studio/models",
    log_path="/teamspace/studios/this_studio/server.log",
    load_audio_vae=True,
    offload_to_cpu=False,
    wait_timeout=300,
    diffusion_fa=False
)

# Display recent logs to verify model status
print("\n--- Recent Engine Logs ---")
print(tail_logs(log_path="/teamspace/studios/this_studio/server.log"))

### 💻 Step 4: Launch Web Interface

Run this cell to launch the Gradio room and generate videos. This will output a public share link (e.g. `*.gradio.live`) to open in a new tab.

In [ ]:
from src.ui import launch

# Launches interface with a shareable public URL
launch()